# Memory

- 챗봇이 대화 내용 히스토리를 관리하기 위한 전략

## 0. CversationBufferMemory

- pros: 기억을 다함
- cons: 기억을 다해서 매번 히스토리를 다 보내야하니 계속 커짐 (비용 청구 ㅎ)
- refs: https://wikidocs.net/233801

In [1]:
from langchain.memory import ConversationBufferMemory

# return_messages True 시 HumanMessage, AiMessage 객체로 반환
# > ChatModel 이 사용할 수 있는 형태
window_memory = ConversationBufferMemory(return_messages=True)

# langchains 0.2v 부터 dictionary 아규잉이 아니라 inputs, ouputs 정식 변수 명이 생긴듯
window_memory.save_context(
    inputs={"human": "Hi!"},
    outputs={"ai": "How are you"}
)
window_memory.load_memory_variables({})

{'history': [HumanMessage(content='Hi!'), AIMessage(content='How are you')]}

## 2. ConversationBufferWindowMemory

- pros: 최근 k 개 만의 상호작용 히스토리만 저장
- cons: 최근 데이터만 저장한다는 것이 단점
- refs: https://wikidocs.net/233800

In [2]:
from langchain.memory import ConversationBufferWindowMemory

buffer_window_memory = ConversationBufferWindowMemory(
    return_messages=True,
    k=5, # 최신 저장할 컨텍스트 수
)

# 10번의 컨텍스트 저장 시도
for i in range(10):
    buffer_window_memory.save_context(
        inputs={"human": str(i)},
        outputs={"ai": str(i)}
    )
    
memory_variables = buffer_window_memory.load_memory_variables({})

# 하지만 5번의 컨텍스트만 저장됨 (human + ai 가 각 컨텍스트 별로 하나씩 존재하니 총 message 는 10개
print(
    len(memory_variables.get('history')), 
    memory_variables.get('history')
)

10 [HumanMessage(content='5'), AIMessage(content='5'), HumanMessage(content='6'), AIMessage(content='6'), HumanMessage(content='7'), AIMessage(content='7'), HumanMessage(content='8'), AIMessage(content='8'), HumanMessage(content='9'), AIMessage(content='9')]


## 3. ConversaionSummaryMemory
- pros: 대화 내용을 자체적으로 요약해서 저장 (그래서 모델이 필요)
- cons: 처음엔 기존 SummaryMemory 보다 더 많은 토큰과 저장공간을 차지
- refs: https://wikidocs.net/233810

In [3]:
from config.langchain import setup_langchain_openai_envs

setup_langchain_openai_envs()

In [4]:
from langchain_openai import ChatOpenAI
from typing import Any
from langchain.memory.chat_memory import BaseChatMemory
from langchain.memory import ConversationSummaryMemory

class MyLlmMemory:
    def __init__(self, memory: BaseChatMemory):
        self.memory = memory
    
    def add_message(self, human_message: str, ai_message: str):
        self.memory.save_context(
            inputs={"human": human_message},
            outputs={"ai": ai_message}
        )
    
    def get_history(self) -> dict[str, Any]:
        return self.memory.load_memory_variables({})


llm = ChatOpenAI(temperature=0.1)
summary_memory = ConversationSummaryMemory(llm=llm, return_messages=True)
my_summary_memory = MyLlmMemory(summary_memory)

In [5]:
my_summary_memory.add_message(
    human_message="안녕 난 대훈이야, 한국에 살",
    ai_message="오 좀 쩌는데?"
)
my_summary_memory.get_history()

{'history': [SystemMessage(content='The human introduces themselves as Daehoon from Korea. The AI responds positively.')]}

In [6]:
my_summary_memory.add_message(
    human_message="한국이 좀 대단하긴하쥐",
    ai_message="오 나도 한번 가고싶은데?"
)
my_summary_memory.get_history()

{'history': [SystemMessage(content='The human introduces themselves as Daehoon from Korea and compliments Korea. The AI expresses interest in visiting Korea.')]}

## 4. ConversationSummaryBufferMemory
> ConversationBufferMemory = ConversationSummaryMemory

- pros: 최근 컨텍스트의 max toeken 수를 저장하면서, 옛 컨텍스트는 요약함
- cons: 처음엔 기존 SummaryMemory 보다 더 많은 토큰과 저장공간을 차지
- refs: https://wikidocs.net/233810

In [7]:
from langchain.memory import ConversationSummaryBufferMemory

llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0.1,
)

my_summary_memory = MyLlmMemory(
    memory=ConversationSummaryBufferMemory(
        llm=llm,
        max_token_length=150,
        return_messages=True
    ),
)

In [8]:
my_summary_memory.add_message(
    human_message="안녕 난 대훈이야, 한국에 살아.",
    ai_message="오 좀 쩌는데?"
)
my_summary_memory.get_history()

{'history': [HumanMessage(content='안녕 난 대훈이야, 한국에 살아.'),
  AIMessage(content='오 좀 쩌는데?')]}

In [9]:
my_summary_memory.add_message(
    human_message="한국이 좀 대단하긴하쥐",
    ai_message="오 나도 한번 가고싶은데?"
)
my_summary_memory.get_history()

{'history': [HumanMessage(content='안녕 난 대훈이야, 한국에 살아.'),
  AIMessage(content='오 좀 쩌는데?'),
  HumanMessage(content='한국이 좀 대단하긴하쥐'),
  AIMessage(content='오 나도 한번 가고싶은데?')]}

In [10]:
my_summary_memory.add_message(
    human_message="대구는 어때? 딱히 할건 없는데 뭐 뭉티기는 맛있어.",
    ai_message="니가 뭉티기에 대해서 뭘아니. 내가 뭉티이기의 진수를 보여주지. 잠깐만 기다려봐"
)
my_summary_memory.get_history()

{'history': [HumanMessage(content='안녕 난 대훈이야, 한국에 살아.'),
  AIMessage(content='오 좀 쩌는데?'),
  HumanMessage(content='한국이 좀 대단하긴하쥐'),
  AIMessage(content='오 나도 한번 가고싶은데?'),
  HumanMessage(content='대구는 어때? 딱히 할건 없는데 뭐 뭉티기는 맛있어.'),
  AIMessage(content='니가 뭉티기에 대해서 뭘아니. 내가 뭉티이기의 진수를 보여주지. 잠깐만 기다려봐')]}

In [11]:
my_summary_memory.add_message(
    human_message="얼마나 기다려야하는데? AI 가 요즘 갑이네 갑이야.",
    ai_message="니가 나의 연산에 대해 뭘 아니? 조금 더 딱기다려봐 뒤적거리고 있으니깐"
)
my_summary_memory.get_history()

{'history': [HumanMessage(content='안녕 난 대훈이야, 한국에 살아.'),
  AIMessage(content='오 좀 쩌는데?'),
  HumanMessage(content='한국이 좀 대단하긴하쥐'),
  AIMessage(content='오 나도 한번 가고싶은데?'),
  HumanMessage(content='대구는 어때? 딱히 할건 없는데 뭐 뭉티기는 맛있어.'),
  AIMessage(content='니가 뭉티기에 대해서 뭘아니. 내가 뭉티이기의 진수를 보여주지. 잠깐만 기다려봐'),
  HumanMessage(content='얼마나 기다려야하는데? AI 가 요즘 갑이네 갑이야.'),
  AIMessage(content='니가 나의 연산에 대해 뭘 아니? 조금 더 딱기다려봐 뒤적거리고 있으니깐')]}

## 5. ConversationKGMemory
> KG: Knowledge Graph

- 지식 그래프가 뭔지, 이해가 안가서 더 공부해봐야 알듯

## 5.7 LCEL Based Memory


In [19]:
class OrdiLlmMemory:
    def __init__(self, memory: BaseChatMemory):
        self.memory = memory
    
    def add_message(self, human_message: str, ai_message: str):
        self.memory.save_context(
            inputs={"human": human_message},
            outputs={"ai": ai_message}
        )
    
    def add_history(self, question, result_content):
        self.memory.save_context(
            inputs={"human": question},
            outputs={"ai": result_content}
        )
    
    def get_history(self, _) -> dict[str, Any]:
        return self.memory.load_memory_variables({})["history"]

memory = OrdiLlmMemory(memory=ConversationBufferMemory(return_messages=True))

In [20]:
from langchain_core.runnables import RunnablePassthrough
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

llm = ChatOpenAI(model="gpt-3.5-turbo")
prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "You are a helpful AI talking to a human"),
        MessagesPlaceholder(variable_name="history"),
        ("human", "{question}"),
    ]
)

chain = RunnablePassthrough.assign(history=memory.get_history) | prompt | llm

In [21]:
def invoke_chain(question):
    result = chain.invoke({"question": question})
    memory.add_history(
        question=question,
        result_content=result.content,
    )
    return result

In [22]:
invoke_chain("나는 대훈이야")

AIMessage(content='안녕하세요, 대훈님! 무엇을 도와드릴까요?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 28, 'prompt_tokens': 28, 'total_tokens': 56}, 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'finish_reason': 'stop', 'logprobs': None}, id='run-aac4ee92-0d9e-4a82-8f10-3d7a4a2f922b-0', usage_metadata={'input_tokens': 28, 'output_tokens': 28, 'total_tokens': 56})

In [23]:
invoke_chain("내 이름은 뭐야?")

AIMessage(content='죄송합니다, 이 채팅창에서는 이름이 제공되지 않습니다. 그래서 제가 이름을 알 수 없어요. 어떻게 도와드릴까요?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 56, 'prompt_tokens': 71, 'total_tokens': 127}, 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'finish_reason': 'stop', 'logprobs': None}, id='run-2b98f323-1c7e-4ff8-a195-7279c19e4bda-0', usage_metadata={'input_tokens': 71, 'output_tokens': 56, 'total_tokens': 127})

In [24]:
invoke_chain("아니 내가 방금 소개로 말한 이름이 뭐였냐고")


AIMessage(content='죄송합니다, 대훈이라고 말씀하셨죠! 어떻게 도와드릴까요, 대훈님?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 49, 'prompt_tokens': 159, 'total_tokens': 208}, 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'finish_reason': 'stop', 'logprobs': None}, id='run-436482bd-19f8-464a-933d-660a2c6dc286-0', usage_metadata={'input_tokens': 159, 'output_tokens': 49, 'total_tokens': 208})